# MIMIC-IV Demo — Guided Exploration

100-patient de-identified EHR dataset from Beth Israel Deaconess Medical Center (BIDMC).

The dataset is split into two modules:
- **`hosp/`** — everything that happens at the hospital level (admissions, labs, diagnoses, meds, billing)
- **`icu/`** — everything recorded inside the ICU (vitals, drips, ventilator settings, fluid balance)

Every table is linked by one or both of:
- `subject_id` — unique per **patient** (person), never changes across visits
- `hadm_id` — unique per **hospital admission** (one patient can have many admissions)
- `stay_id` — unique per **ICU stay** (one admission can have multiple ICU stays)

In [1]:
import pandas as pd

DATA = "mimic-iv-clinical-database-demo-2.2"
HOSP = f"{DATA}/hosp"
ICU  = f"{DATA}/icu"

pd.set_option("display.max_columns", None)  # show all columns
pd.set_option("display.max_colwidth", 60)   # don't truncate text

---
## 1. Patient Demographics

### Table: `patients`

One row per **patient**. This is your master patient list.

| Column | What it means |
|---|---|
| `subject_id` | Unique patient ID — the primary key you use to link all other tables |
| `gender` | Biological sex recorded at admission (`M` / `F`) |
| `anchor_age` | **The patient's age at `anchor_year`** — this is how MIMIC encodes age without storing a real birthdate (privacy) |
| `anchor_year` | A shifted calendar year used as the reference point for `anchor_age`. The real year is hidden; only the *relative* gap between events is preserved |
| `anchor_year_group` | A 3-year band (e.g. `2011 - 2013`) telling you roughly *when* in real time this patient was treated |
| `dod` | Date of death, if the patient died (also shifted to match the de-identified timeline). Blank = alive or unknown |

> **Key de-identification note:** Real birthdates are removed. To estimate a patient's age at a specific admission, calculate how many years passed between `anchor_year` and the admission year, then add that to `anchor_age`.

In [2]:
patients = pd.read_csv(f"{HOSP}/patients.csv.gz")
print(f"Rows: {len(patients)}  |  Columns: {list(patients.columns)}")
patients.head(10)

Rows: 100  |  Columns: ['subject_id', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod']


,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10014729,F,21,2125,2011 - 2013,NaN
1,10003400,F,72,2134,2011 - 2013,2137-09-02
2,10002428,F,80,2155,2011 - 2013,NaN
3,10032725,F,38,2143,2011 - 2013,2143-03-30
4,10027445,F,48,2142,2011 - 2013,2146-02-09
5,10037928,F,78,2175,2011 - 2013,NaN
6,10001725,F,46,2110,2011 - 2013,NaN
7,10040025,F,64,2143,2011 - 2013,2148-02-07
8,10008454,F,26,2110,2011 - 2013,NaN
9,10020640,F,91,2153,2011 - 2013,2154-02-04


### Table: `admissions`

One row per **hospital admission**. A patient can appear multiple times if they were admitted more than once.

| Column | What it means |
|---|---|
| `subject_id` | Links back to `patients` |
| `hadm_id` | Unique ID for this specific hospital stay |
| `admittime` | Datetime the patient was admitted |
| `dischtime` | Datetime the patient was discharged |
| `deathtime` | Datetime of in-hospital death (blank if survived) |
| `admission_type` | Clinical urgency: `URGENT`, `ELECTIVE`, `EU OBSERVATION`, `DIRECT EMER.`, etc. |
| `admit_provider_id` | De-identified ID of the admitting provider |
| `admission_location` | Where the patient came from (e.g. `EMERGENCY ROOM`, `TRANSFER FROM HOSPITAL`) |
| `discharge_location` | Where the patient went after discharge (e.g. `HOME`, `SKILLED NURSING FACILITY`, `DIED`) |
| `insurance` | Payer type: `Medicare`, `Medicaid`, `Other` |
| `language` | Patient's preferred language |
| `marital_status` | Marital status at admission |
| `race` | Self-reported race/ethnicity |
| `edregtime` | When the patient registered in the ED (if they came through the ED) |
| `edouttime` | When the patient left the ED |
| `hospital_expire_flag` | `1` = patient died during this admission, `0` = survived |

In [3]:
admissions = pd.read_csv(f"{HOSP}/admissions.csv.gz", parse_dates=["admittime", "dischtime"])
print(f"Rows: {len(admissions)}")
admissions.head()

Rows: 275


,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,ENGLISH,SINGLE,BLACK/CAPE VERDEAN,2196-02-24 12:15:00,2196-02-24 17:07:00,0
1,10009628,25926192,2153-09-17 17:08:00,2153-09-25 13:20:00,NaN,URGENT,P41R5N,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Medicaid,?,MARRIED,HISPANIC/LATINO - PUERTO RICAN,NaN,NaN,0
2,10018081,23983182,2134-08-18 02:02:00,2134-08-23 19:35:00,NaN,URGENT,P233F6,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicare,ENGLISH,MARRIED,WHITE,2134-08-17 16:24:00,2134-08-18 03:15:00,0
3,10006053,22942076,2111-11-13 23:39:00,2111-11-15 17:20:00,2111-11-15 17:20:00,URGENT,P38TI6,TRANSFER FROM HOSPITAL,DIED,Medicaid,ENGLISH,NaN,UNKNOWN,NaN,NaN,1
4,10031404,21606243,2113-08-04 18:46:00,2113-08-06 20:57:00,NaN,URGENT,P07HDB,TRANSFER FROM HOSPITAL,HOME,Other,ENGLISH,WIDOWED,WHITE,NaN,NaN,0


### Computing Age at Admission

Since real birthdates are removed, we reconstruct age at each admission using:

```
age_at_admission = anchor_age + (admission_year - anchor_year)
```

**Note:** Patients aged 89+ at anchor_year are all recorded as exactly `91` in MIMIC to prevent re-identification of very elderly patients.

In [4]:
# Merge patients onto admissions so we have anchor_age and anchor_year alongside each visit
demo = admissions.merge(patients[["subject_id", "gender", "anchor_age", "anchor_year", "anchor_year_group", "dod"]],
                        on="subject_id", how="left")

# Calculate age at each admission
demo["admit_year"] = demo["admittime"].dt.year
demo["age_at_admission"] = demo["anchor_age"] + (demo["admit_year"] - demo["anchor_year"])

# Show the demographics table
demographics = demo[["subject_id", "hadm_id", "gender", "age_at_admission",
                      "anchor_year_group", "race", "insurance", "marital_status",
                      "admission_type", "discharge_location", "hospital_expire_flag", "dod"]]
demographics.head(10)

,subject_id,hadm_id,gender,age_at_admission,anchor_year_group,race,insurance,marital_status,admission_type,discharge_location,hospital_expire_flag,dod
0,10004235,24181354,M,47,2014 - 2016,BLACK/CAPE VERDEAN,Medicaid,SINGLE,URGENT,SKILLED NURSING FACILITY,0,NaN
1,10009628,25926192,M,58,2011 - 2013,HISPANIC/LATINO - PUERTO RICAN,Medicaid,MARRIED,URGENT,HOME HEALTH CARE,0,NaN
2,10018081,23983182,M,80,2011 - 2013,WHITE,Medicare,MARRIED,URGENT,SKILLED NURSING FACILITY,0,2134-10-28
3,10006053,22942076,M,52,2014 - 2016,UNKNOWN,Medicaid,NaN,URGENT,DIED,1,2111-11-15
4,10031404,21606243,F,82,2014 - 2016,WHITE,Other,WIDOWED,URGENT,HOME,0,NaN
5,10005817,20626031,M,66,2014 - 2016,WHITE,Medicare,MARRIED,URGENT,HOME HEALTH CARE,0,2135-01-19
6,10019385,20297618,M,44,2014 - 2016,WHITE,Other,MARRIED,URGENT,HOME HEALTH CARE,0,NaN
7,10002495,24982426,M,81,2014 - 2016,UNKNOWN,Medicare,MARRIED,URGENT,SKILLED NURSING FACILITY,0,NaN
8,10038081,20755971,F,63,2014 - 2016,UNKNOWN,Other,SINGLE,URGENT,DIED,1,2115-10-12
9,10019917,22585261,M,44,2014 - 2016,OTHER,Other,SINGLE,URGENT,HOME,0,NaN


In [5]:
# Quick summary statistics for age across all admissions
print("Age at admission — summary:")
print(demographics["age_at_admission"].describe())
print()
print("Gender breakdown:")
print(demographics["gender"].value_counts())
print()
print("Race breakdown:")
print(demographics["race"].value_counts())

Age at admission — summary:
count    275.000000
mean      62.727273
std       14.479352
min       21.000000
25%       53.000000
50%       63.000000
75%       72.000000
max       93.000000
Name: age_at_admission, dtype: float64

Gender breakdown:
gender
M    142
F    133
Name: count, dtype: int64

Race breakdown:
race
WHITE                             170
BLACK/AFRICAN AMERICAN             48
UNKNOWN                            17
HISPANIC/LATINO - CUBAN             9
PORTUGUESE                          7
OTHER                               4
UNABLE TO OBTAIN                    4
BLACK/CAPE VERDEAN                  3
HISPANIC/LATINO - SALVADORAN        3
WHITE - BRAZILIAN                   3
WHITE - OTHER EUROPEAN              2
PATIENT DECLINED TO ANSWER          2
HISPANIC OR LATINO                  2
HISPANIC/LATINO - PUERTO RICAN      1
Name: count, dtype: int64


---
## 2. ICD Codes — Diagnoses and Procedures

ICD (International Classification of Diseases) codes are the standardized codes clinicians and billing teams use to record **what was wrong with the patient** (diagnoses) and **what was done to the patient** (procedures).

MIMIC-IV contains two versions:
- **ICD-9** (`icd_version = 9`) — older coding system, used for stays before ~2015
- **ICD-10** (`icd_version = 10`) — current system, more granular

---
### Table: `diagnoses_icd`

One row per diagnosis code per admission. A single admission typically has many codes.

| Column | What it means |
|---|---|
| `subject_id` | Patient |
| `hadm_id` | Which hospital admission this diagnosis belongs to |
| `seq_num` | **Priority order** — `1` is the **primary (principal) diagnosis** (the main reason for the admission). Higher numbers are secondary/comorbid conditions |
| `icd_code` | The ICD code itself (e.g. `41401` = coronary artery disease) |
| `icd_version` | `9` or `10` |

### Table: `d_icd_diagnoses` (dictionary)

Maps every `icd_code` to a human-readable description.

| Column | What it means |
|---|---|
| `icd_code` | The code (join key) |
| `icd_version` | `9` or `10` |
| `long_title` | Full English description of the diagnosis |

In [6]:
diagnoses    = pd.read_csv(f"{HOSP}/diagnoses_icd.csv.gz", dtype={"icd_code": str})
d_icd_dx     = pd.read_csv(f"{HOSP}/d_icd_diagnoses.csv.gz", dtype={"icd_code": str})

# Attach human-readable names
diagnoses_named = diagnoses.merge(d_icd_dx, on=["icd_code", "icd_version"], how="left")

print(f"Total diagnosis rows: {len(diagnoses_named)}")
diagnoses_named.head(10)

Total diagnosis rows: 4506


,subject_id,hadm_id,seq_num,icd_code,icd_version,long_title
0,10035185,22580999,3,4139,9,Other and unspecified angina pectoris
1,10035185,22580999,10,V707,9,Examination of participant in clinical trial
2,10035185,22580999,1,41401,9,Coronary atherosclerosis of native coronary artery
3,10035185,22580999,9,3899,9,Unspecified hearing loss
4,10035185,22580999,11,V8532,9,"Body Mass Index 32.0-32.9, adult"
5,10035185,22580999,2,25002,9,"Diabetes mellitus without mention of complication, type ..."
6,10035185,22580999,7,71696,9,"Arthropathy, unspecified, lower leg"
7,10035185,22580999,8,V1083,9,Personal history of other malignant neoplasm of skin
8,10035185,22580999,4,2724,9,Other and unspecified hyperlipidemia
9,10035185,22580999,6,V5867,9,Long-term (current) use of insulin


In [7]:
# Primary diagnoses only (seq_num == 1) — the main reason each patient was admitted
primary_dx = diagnoses_named[diagnoses_named["seq_num"] == 1].copy()
print(f"Primary diagnoses: {len(primary_dx)} admissions")
primary_dx[["subject_id", "hadm_id", "icd_code", "icd_version", "long_title"]].head(10)

Primary diagnoses: 275 admissions


,subject_id,hadm_id,icd_code,icd_version,long_title
2,10035185,22580999,41401,9,Coronary atherosclerosis of native coronary artery
14,10009049,22995465,0388,9,Other specified septicemias
24,10009035,28324362,4240,9,Mitral valve disorders
32,10014729,23300884,99859,9,Other postoperative infection
41,10018501,28479513,85221,9,Subdural hemorrhage following injury without mention of ...
56,10001217,24597018,3240,9,Intracranial abscess
65,10026354,24547356,S0262XA,10,"Fracture of subcondylar process of mandible, initial enc..."
71,10023771,20044587,41401,9,Coronary atherosclerosis of native coronary artery
81,10009628,25926192,41401,9,Coronary atherosclerosis of native coronary artery
95,10022017,22342963,I2510,10,Atherosclerotic heart disease of native coronary artery ...


### Table: `procedures_icd`

One row per procedure performed during an admission. These are the major clinical interventions.

| Column | What it means |
|---|---|
| `subject_id` | Patient |
| `hadm_id` | Admission |
| `seq_num` | Priority — `1` is the **primary procedure** (the main intervention) |
| `chartdate` | Date the procedure was performed |
| `icd_code` | ICD procedure code |
| `icd_version` | `9` or `10` |

### Table: `d_icd_procedures` (dictionary)

Same structure as `d_icd_diagnoses` but for procedures.

In [8]:
procedures   = pd.read_csv(f"{HOSP}/procedures_icd.csv.gz", dtype={"icd_code": str})
d_icd_proc   = pd.read_csv(f"{HOSP}/d_icd_procedures.csv.gz", dtype={"icd_code": str})

procedures_named = procedures.merge(d_icd_proc, on=["icd_code", "icd_version"], how="left")
print(f"Total procedure rows: {len(procedures_named)}")
procedures_named.head(10)

Total procedure rows: 722


,subject_id,hadm_id,seq_num,chartdate,icd_code,icd_version,long_title
0,10011398,27505812,3,2146-12-15,3961,9,Extracorporeal circulation auxiliary to open heart surgery
1,10011398,27505812,2,2146-12-15,3615,9,Single internal mammary-coronary artery bypass
2,10011398,27505812,1,2146-12-15,3614,9,(Aorto)coronary bypass of four or more coronary arteries
3,10014729,23300884,4,2125-03-23,3897,9,Central venous catheter placement with guidance
4,10014729,23300884,1,2125-03-20,3403,9,Reopening of recent thoracotomy site
5,10014729,23300884,3,2125-03-20,3404,9,Insertion of intercostal catheter for drainage
6,10014729,23300884,2,2125-03-24,3452,9,Thoracoscopic decortication of lung
7,10014729,28889419,1,2125-02-27,3845,9,"Resection of vessel with replacement, thoracic vessels"
8,10014729,28889419,4,2125-02-28,0390,9,Insertion of catheter into spinal canal for infusion of ...
9,10014729,28889419,5,2125-02-28,0391,9,Injection of anesthetic into spinal canal for analgesia


In [9]:
# Primary procedures only
primary_proc = procedures_named[procedures_named["seq_num"] == 1]
primary_proc[["subject_id", "hadm_id", "chartdate", "icd_code", "icd_version", "long_title"]].head(10)

,subject_id,hadm_id,chartdate,icd_code,icd_version,long_title
2,10011398,27505812,2146-12-15,3614,9,(Aorto)coronary bypass of four or more coronary arteries
4,10014729,23300884,2125-03-20,3403,9,Reopening of recent thoracotomy site
7,10014729,28889419,2125-02-27,3845,9,"Resection of vessel with replacement, thoracic vessels"
31,10007818,22987108,2146-06-11,0W9G3ZZ,10,"Drainage of Peritoneal Cavity, Percutaneous Approach"
38,10004235,25970245,2196-06-14,5123,9,Laparoscopic cholecystectomy
44,10004235,24181354,2196-02-24,9671,9,Continuous invasive mechanical ventilation for less than...
47,10026255,20437651,2200-09-17,3409,9,Other incision of pleura
48,10007058,22954658,2167-11-08,02C03ZZ,10,"Extirpation of Matter from Coronary Artery, One Artery, ..."
51,10039831,26924951,2115-12-28,403,9,Regional lymph node excision
53,10018328,23786647,2154-04-24,03VG3DZ,10,Restriction of Intracranial Artery with Intraluminal Dev...


---
## 3. Full Patient Story — Demographics + All ICD Events

Pulling everything together for a single patient: who they are, why they came in, and what was done.

In [10]:
# Pick any subject_id from the list below, or leave as-is to use the first patient
sample_id = patients["subject_id"].iloc[0]
print(f"Exploring patient: {sample_id}")

print("\n=== Demographics ===")
display(demographics[demographics.subject_id == sample_id])

print("\n=== All Diagnoses (sorted by admission then priority) ===")
pat_dx = diagnoses_named[diagnoses_named.subject_id == sample_id].sort_values(["hadm_id", "seq_num"])
display(pat_dx[["hadm_id", "seq_num", "icd_code", "icd_version", "long_title"]])

print("\n=== All Procedures (sorted by date then priority) ===")
pat_proc = procedures_named[procedures_named.subject_id == sample_id].sort_values(["chartdate", "seq_num"])
display(pat_proc[["hadm_id", "seq_num", "chartdate", "icd_code", "icd_version", "long_title"]])

Exploring patient: 10014729

=== Demographics ===


,subject_id,hadm_id,gender,age_at_admission,anchor_year_group,race,insurance,marital_status,admission_type,discharge_location,hospital_expire_flag,dod
51,10014729,23300884,F,21,2011 - 2013,WHITE - OTHER EUROPEAN,Other,SINGLE,EW EMER.,HOME HEALTH CARE,0,NaN
258,10014729,28889419,F,21,2011 - 2013,WHITE - OTHER EUROPEAN,Other,SINGLE,SURGICAL SAME DAY ADMISSION,HOME HEALTH CARE,0,NaN



=== All Diagnoses (sorted by admission then priority) ===


,hadm_id,seq_num,icd_code,icd_version,long_title
32,23300884,1,99859,9,Other postoperative infection
35,23300884,2,5109,9,Empyema without mention of fistula
28,23300884,3,99832,9,Disruption of external operation (surgical) wound
36,23300884,4,51189,9,"Other specified forms of effusion, except tuberculous"
37,23300884,5,6822,9,Cellulitis and abscess of trunk
30,23300884,6,E8782,9,"Surgical operation with anastomosis, bypass, or graft, w..."
34,23300884,7,E8497,9,Accidents occurring in residential institution
39,23300884,8,V151,9,"Personal history of surgery to heart and great vessels, ..."
31,23300884,9,V1204,9,Personal history of Methicillin resistant Staphylococcus...
38,23300884,10,34600,9,"Migraine with aura, without mention of intractable migra..."



=== All Procedures (sorted by date then priority) ===


,hadm_id,seq_num,chartdate,icd_code,icd_version,long_title
7,28889419,1,2125-02-27,3845,9,"Resection of vessel with replacement, thoracic vessels"
11,28889419,2,2125-02-27,3959,9,Other repair of vessel
10,28889419,3,2125-02-27,3961,9,Extracorporeal circulation auxiliary to open heart surgery
8,28889419,4,2125-02-28,0390,9,Insertion of catheter into spinal canal for infusion of ...
9,28889419,5,2125-02-28,0391,9,Injection of anesthetic into spinal canal for analgesia
12,28889419,6,2125-03-03,8605,9,Incision with removal of foreign body or device from ski...
4,23300884,1,2125-03-20,3403,9,Reopening of recent thoracotomy site
5,23300884,3,2125-03-20,3404,9,Insertion of intercostal catheter for drainage
3,23300884,4,2125-03-23,3897,9,Central venous catheter placement with guidance
6,23300884,2,2125-03-24,3452,9,Thoracoscopic decortication of lung


---
## 4. Reference — All Other Tables in MIMIC-IV Demo

### Hospital module (`hosp/`)

| Table | What it contains |
|---|---|
| `patients` | One row per patient — gender, anchor age, death date |
| `admissions` | One row per hospital stay — timing, type, location, insurance, demographics |
| `transfers` | Every time a patient moved between units/wards within a stay |
| `diagnoses_icd` | ICD diagnosis codes billed for each admission |
| `d_icd_diagnoses` | Dictionary: maps ICD diagnosis codes → English descriptions |
| `procedures_icd` | ICD procedure codes billed for each admission |
| `d_icd_procedures` | Dictionary: maps ICD procedure codes → English descriptions |
| `labevents` | Every lab test result (e.g. blood glucose, creatinine, CBC) with value, units, and reference range |
| `d_labitems` | Dictionary: maps lab `itemid` → test name, fluid type, category |
| `prescriptions` | Medication orders — drug name, dose, route, start/stop times |
| `pharmacy` | More detailed pharmacy dispensing records linked to prescriptions |
| `emar` | Electronic Medication Administration Record — when each dose was *actually given* |
| `emar_detail` | Line-level detail for each eMAR entry (e.g. partial doses, waste) |
| `poe` | Provider Order Entry — every order a clinician placed (meds, labs, diets, consults) |
| `poe_detail` | Additional fields for specific order types |
| `microbiologyevents` | Culture results — specimen, organism grown, antibiotic sensitivities |
| `omr` | Outpatient Measurement Record — vitals and measurements captured outside of charted ICU events (e.g. blood pressure, BMI from clinic visits) |
| `services` | Which clinical service was caring for the patient at each point (e.g. Medicine, Surgery, Cardiology) |
| `hcpcsevents` | HCPCS billing codes (outpatient procedure/supply codes, similar to ICD but for billing) |
| `d_hcpcs` | Dictionary: maps HCPCS codes → descriptions |
| `drgcodes` | Diagnosis-Related Group codes — the billing bucket used for Medicare reimbursement |
| `provider` | De-identified provider IDs (no names — just IDs to link orders to caregivers) |

### ICU module (`icu/`)

| Table | What it contains |
|---|---|
| `icustays` | One row per ICU stay — which unit, admit/discharge times, length of stay |
| `chartevents` | The **largest table** — every charted measurement in the ICU: heart rate, blood pressure, temperature, GCS score, ventilator settings, nurse assessments, etc. Recorded continuously throughout the stay |
| `d_items` | Dictionary: maps ICU `itemid` → measurement name, unit, category (links into `chartevents`, `inputevents`, etc.) |
| `inputevents` | Every fluid or medication given IV in the ICU — drug name, rate, amount, start/end time |
| `ingredientevents` | Nutritional content breakdown of IV fluids (calories, protein, etc.) |
| `outputevents` | Fluid outputs — urine, drain output, etc., with timestamps |
| `procedureevents` | ICU-level procedures charted in the flowsheet (e.g. arterial line insertion, dialysis session) |
| `datetimeevents` | Charted timestamps for ICU events that are date/time values rather than numbers (e.g. when a line was placed) |
| `caregiver` | De-identified ICU caregiver IDs linked to chart entries |